# Machine Learning for Thermal Engineers
## Lesson 9: Introduction to Time Series

Author: Srikantan Natarajan

# 1. Problem Motivation

In Lesson 8, we predicted compressor power from a snapshot of conditions:

TempError, SolarLoad, AmbientTemp → CompressorPower

That model treated every data point as independent. It had no memory of what happened before.

But thermal systems evolve over time. A cabin parked in the sun for 45 minutes behaves very differently from one parked for 5 minutes — even if the current ambient temperature and solar load look identical.

This lesson introduces **time series modeling**: learning from data where the sequence and history of readings matter.

# 2. Three New Ideas

**Autocorrelation**

How similar is a signal to a delayed version of itself? For cabin temperature during a solar soak, the reading at minute 10 is very close to the reading at minute 9. The cabin changes slowly because of thermal mass. This is called high autocorrelation — and it means the recent past is a strong predictor of the near future.

**Lag Features**

A lag feature is the value of a variable at a previous time step, added as a new input column.

Instead of:

T_ambient, SolarLoad → T_cabin

We give the model:

T_cabin_1min_ago, T_ambient, SolarLoad → T_cabin

Now the model knows where the cabin temperature was. It can detect whether the system is heating up or cooling down.

**Rolling Windows**

A rolling window computes the average of the last N readings as a new feature. This smooths out sensor noise and captures the recent trend — is the cabin warming or cooling overall?

Together, lag features and rolling windows give the model a sense of time — something a static regression model simply does not have.

# 3. Engineering Analogy

Thermal engineers already think this way when reading test data.

When you look at a temperature trace, you don't just read the value at one instant — you read the trend. Is it still climbing? Has it leveled off? Is it starting to drop?

Lag features and rolling windows give a machine learning model that same awareness of trajectory — in a form it can learn from.

# 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

np.random.seed(42)

# 5. Data — Solar Soak Scenario

We simulate 90 minutes of cabin temperature data:

- Minutes 0–60: solar soak, cabin heats up
- Minutes 60–90: HVAC turns on, cabin cools down

This gives us a time series with a clear trend — a good starting point for understanding why history matters.

In [ ]:
time       = np.arange(0, 90)
T_ambient  = 35 + np.random.normal(0, 0.3, 90)
solar_load = np.where(time < 60, 800, 200)
hvac_on    = np.where(time >= 60, 1, 0)

T_cabin = (
    25
    + 0.35 * np.clip(time, 0, 60)
    - 0.50 * np.clip(time - 60, 0, None)
    + np.random.normal(0, 0.2, 90)
)

df = pd.DataFrame({
    "time_min":   time,
    "T_ambient":  T_ambient,
    "solar_load": solar_load,
    "hvac_on":    hvac_on,
    "T_cabin":    T_cabin
})

df.head()

# 6. Visualization

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df.time_min, df.T_cabin, color="firebrick", linewidth=2, label="Cabin Temp")
plt.axvline(x=60, color="green", linestyle="--", linewidth=1.5, label="HVAC ON")
plt.xlabel("Time (minutes)")
plt.ylabel("Temperature (°C)")
plt.title("Cabin Temperature — Solar Soak + HVAC Cooling")
plt.legend()
plt.grid(True)
plt.show()

# 7. Autocorrelation

We measure how strongly the current cabin temperature is correlated with its value from 1, 5, and 10 minutes earlier.

A value of 1.0 means perfectly predictable from the past. A value near 0 means the past tells us nothing.

In [ ]:
for lag in [1, 5, 10]:
    r = df["T_cabin"].autocorr(lag=lag)
    print(f"Autocorrelation at lag {lag:2d} min: r = {r:.4f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(df["T_cabin"].shift(1), df["T_cabin"], alpha=0.5, color="steelblue")
plt.xlabel("T_cabin — 1 minute ago (°C)")
plt.ylabel("T_cabin — now (°C)")
plt.title("Autocorrelation: Now vs 1 Minute Ago")
plt.grid(True)
plt.show()

print("The tight diagonal confirms: where the cabin was 1 minute ago")
print("is a very strong predictor of where it is now.")

# 8. Creating Lag Features and a Rolling Window

We add three new columns:

- T_cabin_lag1 → cabin temperature 1 minute ago
- T_cabin_lag5 → cabin temperature 5 minutes ago
- T_cabin_roll5 → average cabin temperature over the last 5 minutes

In [ ]:
df["T_cabin_lag1"]  = df["T_cabin"].shift(1)
df["T_cabin_lag5"]  = df["T_cabin"].shift(5)
df["T_cabin_roll5"] = df["T_cabin"].shift(1).rolling(window=5).mean()

df = df.dropna().reset_index(drop=True)

df[["time_min", "T_cabin", "T_cabin_lag1", "T_cabin_lag5", "T_cabin_roll5"]].head(10)

# 9. Model Training — Static vs Time-Aware

We train two Ridge regression models and compare them:

- **Model A (static):** T_ambient, solar_load, hvac_on only
- **Model B (time-aware):** same inputs plus lag features and rolling window

Both models predict T_cabin one minute into the future.

Important: for time series, we never shuffle the train/test split. We train on earlier data and test on later data — the same way the model would be used in practice.

In [ ]:
df["target"] = df["T_cabin"].shift(-1)
df = df.dropna()

split   = int(0.7 * len(df))
y_train = df["target"].values[:split]
y_test  = df["target"].values[split:]

# Model A: static features only
cols_A   = ["T_ambient", "solar_load", "hvac_on"]
scaler_A = StandardScaler()
model_A  = Ridge(alpha=1.0)
model_A.fit(scaler_A.fit_transform(df[cols_A].values[:split]), y_train)
pred_A   = model_A.predict(scaler_A.transform(df[cols_A].values[split:]))

print("Model A (static) MAE:", round(mean_absolute_error(y_test, pred_A), 3), "°C")

In [ ]:
# Model B: static + lag + rolling features
cols_B   = ["T_ambient", "solar_load", "hvac_on", "T_cabin_lag1", "T_cabin_lag5", "T_cabin_roll5"]
scaler_B = StandardScaler()
model_B  = Ridge(alpha=1.0)
model_B.fit(scaler_B.fit_transform(df[cols_B].values[:split]), y_train)
pred_B   = model_B.predict(scaler_B.transform(df[cols_B].values[split:]))

mae_A = mean_absolute_error(y_test, pred_A)
mae_B = mean_absolute_error(y_test, pred_B)

print("Model A (static)     MAE:", round(mae_A, 3), "°C")
print("Model B (time-aware) MAE:", round(mae_B, 3), "°C")
print("Improvement:", round(mae_A - mae_B, 3), "°C")

# 10. Visualization

In [ ]:
time_test = df["time_min"].values[split:]

plt.figure(figsize=(10, 4))
plt.plot(time_test, y_test,  color="firebrick", linewidth=2,   label="Actual")
plt.plot(time_test, pred_A,  color="steelblue", linewidth=1.5, linestyle="--", label=f"Model A — Static (MAE = {mae_A:.2f}°C)")
plt.plot(time_test, pred_B,  color="darkgreen", linewidth=1.5, linestyle="--", label=f"Model B — Time-Aware (MAE = {mae_B:.2f}°C)")
plt.xlabel("Time (minutes)")
plt.ylabel("Cabin Temperature (°C)")
plt.title("Static vs Time-Aware Model — 1-Minute Ahead Prediction")
plt.legend()
plt.grid(True)
plt.show()

# 11. Engineering Interpretation

The time-aware model outperforms the static model because cabin temperature is a slow-moving signal driven by thermal mass.

Where the cabin was 1 minute ago is a stronger predictor of where it will be next minute than the current ambient temperature or solar load alone.

The rolling average adds further value by smoothing out sensor noise and capturing the direction of the trend — not just the most recent reading.

# 12. Limitations

- This model uses synthetic data with simplified dynamics
- Lag features work well for 1-step ahead prediction but accumulate error over longer horizons
- Real cabin temperature is influenced by occupant presence, door events, and solar angle — none of which are captured here
- Ridge regression with lag features is a starting point — not a production forecasting model

Future lessons will address longer-horizon forecasting with ARIMA (Lesson 10) and sequence models like LSTM (Lesson 12).

# 13. Key Takeaway

Thermal systems have memory. The past state of the system carries information that a snapshot of current conditions alone cannot provide.

- Autocorrelation measures how much the past predicts the future
- Lag features give a model access to recent history
- Rolling windows capture trend and reduce noise
- A time-aware model consistently outperforms a static model on sequential thermal data
- Always split time series by time — never shuffle